# COGS 188 - Project Proposal

# Names

- Charis Chi
- Krasimira Magdadaro
- Ran Yao

**GitHub Repository:** https://github.com/charisscchi/COGS188-Final-Project

# Abstract

Our project aims to investigate whether traffic-aware route planning can reduce passenger pickup waiting times in a simulated robotaxi system inspired by Tesla's Cybercab. We plan to use New York City taxi trip records containing trip times and pickup and drop-off locations, along with traffic speed data and, if available, traffic volume measurements collected at different locations and times. Using OpenStreetMap to construct the road network and SUMO to simulate traffic conditions, we will compare three approaches: static A*, traffic-aware A*, and an ε-greedy learning method. We will use the data to create realistic traffic scenarios and examine how different routing strategies affect the time it takes for an assigned taxi to reach a passenger. We will evaluate their performance using simulated passenger pickup waiting time, travel time, computation time, and pickup completion rate. Our goal is to determine whether incorporating traffic information and learning from previous simulated trips can improve routing efficiency compared with traditional static route planning.

# Background

Beginning in the early 2000s and 2010s, a new form of rideshare launched which allowed for drivers to provide transportation using personal vehicles as well as connecting to passengers through an app similar to a carpool. This led to better accessibility for both drivers looking to make extra money and passengers in need of easy transportation. This transformed the way people got around in large cities where “By 2015, Uber had more than 162,000 drivers, and had operations in many US cities as well as sixty countries around the world…” while competing platform “Lyft had over 60,000 drivers throughout the United States.” [1] One such solution that independent drivers have tackled is the wide availability of transportation anywhere one may be. Through these platforms, drivers are able to find and pick up passengers anywhere accessible by a car, as opposed to typical forms of public transportation with specific systems, stops, and wait times. As accessible as ride sharing may be, however, these drivers are still subject to obstacles on the road such as traffic congestion and relative location to passengers in need, especially in bigger cities. It was found that “In New York City, there are approximately 68,000 Uber and Lyft cars, about five times as many as yellow taxis, which has caused average speeds during business hours in Manhattan’s core to drop to a crawl...”[2] As a result, there is still a ways to go towards making ridesharing more efficient. One finding suggests improving distributions of drivers throughout the city so that each area may have a number of drivers within a short distance of a passenger. Through a guide, a repositioning heatmap “...highlight the driver-specific earning opportunities, approximated based on their travel time to areas with a shortage of drivers to satisfy the expected demand in the near future.”[3] Through this guide for drivers, the heatmap aims to direct drivers in a more efficient distribution so that they may have access to more pickup and earning opportunities. In turn, this also improves the chances that a passenger is within easy access to a driver, thus reducing wait times. 
A* is one approach to route planning because it uses heuristic search to find a minimum-cost path through a graph. Hart, Nilsson, and Raphael (1968) studied the theoretical basis of heuristic search and its ability to find minimum-cost paths [4]. However, a route based on fixed travel-time estimates may not remain the fastest when traffic conditions change. This is why our project compares static A* with traffic-aware A* and an ε-greedy route-selection method to investigate whether using traffic information can reduce passenger pickup waiting times.

# Problem Statement

One challenge for autonomous taxi services is getting to passengers quickly, especially when traffic conditions change throughout the day. A route that looks like the shortest or fastest option may take longer during rush hour or unexpected congestion. If a taxi follows a route without considering current traffic conditions, passengers may end up waiting longer than necessary.

Our project aims to investigate whether traffic-aware route planning can reduce passenger pickup waiting times compared with traditional static A* search. We will focus on the route an assigned taxi takes from its starting location to the passenger's pickup location, rather than changing which taxi gets assigned. Using a simulated New York City road network, we plan to compare three approaches: static A*, traffic-aware A*, and an ε-greedy learning method that selects among candidate routes based on previous simulated trips.

We will represent the road network as a graph, where intersections are nodes and road segments are edges with estimated travel-time costs. We will use real-world NYC taxi trip and traffic data to help create realistic pickup scenarios and traffic conditions. For each simulated passenger request, pickup waiting time will be calculated as the difference between the passenger's request time and the taxi's arrival time at the pickup location. Our main goal is to minimize the average pickup waiting time across multiple requests. Since the historical taxi records do not provide the locations of all available taxis, we will generate taxi starting positions in the simulation using a consistent procedure across experiments.

To compare the approaches fairly, we will test them using the same passenger requests, assigned taxis, and traffic scenarios in SUMO. We will measure average pickup waiting time, travel time, route computation time, and pickup completion rate. We will also test different values of ε to examine how exploration affects the learning-based method. By repeating the experiments under controlled conditions and recording the simulation settings and random seeds, we can compare the results and evaluate whether traffic-aware routing improves pickup efficiency.

# Data

### Data Set 1: NYC DOT Traffic Speeds

**Link/reference:**

https://data.cityofnewyork.us/Transportation/DOT-Traffic-Speeds/i4gi-tjb9/about_data

**Dataset size:**
- Approximately 113 million observations (rows)
- 13 variables (columns)

**What an observation consists of:**

Each observation represents a traffic speed and travel-time measurement for a monitored road link at a particular timestamp, rather than the speed of an individual vehicle.

**Important variables and how they are represented:**

- `link_points`: Coordinates showing the location of a monitored road link.
- `speed`: Average measured traffic speed on the road link.
- `travel_time`: Estimated time needed to travel along the road link.
- `data_as_of`: Timestamp indicating when the traffic measurement was received.
- `link_id`: Identifier for the monitored road link.
- `borough`: NYC borough where the road link is located.

**Special handling, transformations, and cleaning:**

We will filter traffic observations to the selected geographic area and a time period that overlaps with the taxi trip data. We will convert recorded speeds and travel times into numeric values, remove invalid or missing observations, and match monitored road links to the OpenStreetMap network using their geographic coordinates. Because traffic measurements are unavailable for some roads, we will use a consistent fallback travel-time estimate for unmatched road segments. We will document the matching coverage and any assumptions used in the simulation.

### Data Set 2: NYC High Volume FHV Trip Records (2024)

**Link/reference:**

https://data.cityofnewyork.us/Transportation/2024-High-Volume-FHV-Trip-Data/bqxa-629f

**Dataset size:**
- Approximately 239 million observations (rows) for the full year of 2024.
- 24 variables (columns).

**What an observation consists of:**

Each observation represents one recorded trip made by a high-volume for-hire vehicle (FHV). We plan to select one month of records from 2024 and use a manageable subset for our simulation experiments.

**Important variables and how they are represented:**

- `request_datetime`: Date and time when the passenger requested a pickup.
- `pickup_datetime`: Date and time when the passenger was picked up.
- `dropoff_datetime`: Date and time when the passenger was dropped off.
- `PULocationID`: Numeric ID of the NYC taxi zone where the passenger was picked up.
- `DOLocationID`: Numeric ID of the NYC taxi zone where the passenger was dropped off.
- `trip_miles`: Trip distance in miles.
- `trip_time`: Trip duration in seconds.

**Special handling, transformations, and cleaning:**

We plan to select one month of 2024 High Volume FHV trip records and use a manageable subset for our experiments. We will convert request, pickup, and drop-off timestamps into a consistent datetime format and remove records with missing locations, invalid timestamps, or unrealistic trip durations. Since pickup and drop-off locations are represented by taxi zone IDs rather than exact GPS coordinates, we will use the TLC taxi zone boundaries and OpenStreetMap road network to generate representative pickup locations for the simulation. Available taxi starting locations will be simulated because the dataset does not record the positions of all unoccupied taxis.

# Proposed Solution

We propose a route-planning system for a simulated autonomous taxi traveling from its assigned starting location to a passenger's pickup location in New York City. We will represent the road network as a graph, where intersections are nodes and road segments are edges with estimated travel-time costs. We plan to use NYC High Volume FHV trip records to identify passenger request times and pickup areas, OpenStreetMap to construct the road network, and NYC traffic speed data to estimate travel times on monitored road segments. Since the datasets do not provide the locations of all available taxis, we will simulate their starting positions and keep taxi assignments the same across all routing methods.

We will compare the following methods:

**1. Static A* baseline:** We will use A* search to find a route based on estimated travel times under uncongested conditions. This route will be selected before the pickup attempt and will not use current traffic information.

**2. Traffic-aware route planning:** We will use A* with road costs based on available traffic speed or travel-time information. For road segments without matching traffic data, we will use a consistent fallback estimate. This method may avoid slower monitored roads because it considers traffic conditions when assigning costs to routes.

**3. ε-greedy route selection:** For a limited set of repeated taxi starting locations and passenger pickup areas, we will generate several candidate routes. Each candidate route will be treated as an action. After each simulated pickup attempt, the algorithm will receive a reward based on the time required for the taxi to reach the passenger, with shorter pickup waiting times receiving higher rewards. The algorithm will update its estimated value for the selected route and use an ε-greedy policy: it will usually choose the route with the highest estimated value while sometimes exploring another route. We will test several values of ε to examine the effect of exploration.

The bandit method may work because repeated simulated pickup attempts provide feedback about how well each candidate route performs under similar conditions. However, its estimates may not generalize to unfamiliar pickup locations or traffic conditions, so we will initially evaluate it on a limited set of repeated pickup scenarios.

We plan to use SUMO to simulate taxi movements from assigned starting locations to passenger pickup locations. For each simulated request, we will record the passenger's request time, the taxi's departure time, and its arrival time at the pickup location. The difference between the request time and arrival time will represent simulated passenger pickup waiting time. We will compare all three methods using the same passenger requests, assigned taxis, and background traffic scenarios, with separate simulation runs for parameter selection and final evaluation.

We will evaluate average passenger pickup waiting time, route travel time, traffic waiting time, computation time, and pickup completion rate. If suitable acceleration data can be generated, we may also examine ride smoothness as an additional metric.

To make the experiment reproducible, we will record the selected geographic area, data period, traffic scenarios, candidate routes, ε values, random seeds, and software versions. Before the full experiment, we will test whether the NYC traffic data can be matched to the road network. If the match is insufficient, we will use simulated traffic scenarios and clearly report that result.


# Evaluation Metrics

We will evaluate static A*, traffic-aware A*, and ε-greedy route selection using the same passenger requests, assigned taxi starting locations, and simulated traffic conditions. Our primary metric will be passenger pickup waiting time, since our main goal is to determine whether traffic-aware routing helps taxis reach passengers faster. We will also compare route travel time, traffic waiting time, computation time, and pickup completion rate to evaluate the overall efficiency and reliability of each approach.

### 1. Average Passenger Pickup Waiting Time

Passenger pickup waiting time measures how long a passenger waits between requesting a taxi and the assigned taxi arriving at their pickup location. For each completed pickup, we calculate:

$$
W_i^{\text{pickup}} = t_i^{\text{arrival}} - t_i^{\text{request}}
$$

The average pickup waiting time across $N$ completed pickups is:

$$
\overline{W}_{\text{pickup}} = \frac{1}{N}\sum_{i=1}^{N} W_i^{\text{pickup}}
$$

where $t_i^{\text{request}}$ is the passenger's request time and $t_i^{\text{arrival}}$ is when the assigned taxi reaches the pickup location. A lower average means passengers spend less time waiting for their taxis. These times will be measured in our simulation.

### 2. Average Route Travel Time

Route travel time measures how long the assigned taxi takes to travel from its starting location to the passenger's pickup location. For each completed pickup, it is calculated as the arrival time minus the taxi's departure time.

$$
T_i^{\text{route}} = t_i^{\text{arrival}} - t_i^{\text{departure}}
$$

The average route travel time is:

$$
\overline{T}_{\text{route}} = \frac{1}{N}\sum_{i=1}^{N} T_i^{\text{route}}
$$

A lower value indicates that the routing algorithm helps taxis reach pickup locations more quickly. If taxis depart immediately after a passenger request, route travel time and passenger pickup waiting time will be equal. We will report this assumption when analyzing the results.

### 3. Average Traffic Waiting Time

Traffic waiting time measures how much time a taxi spends nearly stopped during its route because of traffic conditions. We plan to use SUMO's waiting-time output, which counts time when a vehicle's speed is at or below 0.1 m/s, excluding scheduled stops.

The average traffic waiting time is:

$$
\overline{W}_{\text{traffic}} = \frac{1}{N}\sum_{i=1}^{N} W_i^{\text{traffic}}
$$

where $W_i^{\text{traffic}}$ is the traffic waiting time for completed pickup attempt $i$.

A lower value indicates less time spent nearly stopped during the route. This metric is different from passenger pickup waiting time and helps us understand how congestion affects each routing method.

### 4. Average Computation Time

Computation time measures how long an algorithm takes to select a route. We will use the same timing procedure for each method and calculate:

$$
\overline{C} = \frac{1}{M}\sum_{i=1}^{M} C_i
$$

where $C_i$ is the route-selection time for attempt $i$, and $M$ is the total number of route-planning attempts.

This metric allows us to compare routing performance with the computational cost of selecting a route. For the ε-greedy learning method, we will report training time separately from route-selection time.

### 5. Pickup Completion Rate

Pickup completion rate measures the percentage of attempted passenger pickups in which the assigned taxi successfully reaches the passenger's pickup location within the simulation time limit.

$$
\text{Pickup Completion Rate} =
\frac{\text{Number of completed pickups}}
{\text{Number of attempted pickups}}
\times 100\%
$$

A higher completion rate indicates that the algorithm successfully handles more pickup requests. This metric also prevents an approach from appearing better simply because difficult or unfinished pickup attempts are excluded from its average travel time.

### Additional Evaluation

We will test different values of ε for the learning-based method to examine how exploration affects its performance. We will use separate simulation runs for parameter selection and final evaluation, while keeping passenger requests, taxi assignments, and traffic scenarios comparable across methods.

If feasible, we may also examine simulated acceleration and braking as an additional measure of ride smoothness. However, we will not treat these measurements as direct evidence of passenger comfort or satisfaction without feedback from actual passengers.


# Ethics & Privacy

Our project uses publicly available NYC taxi trip records and traffic data to evaluate route-planning methods in a simulated robotaxi system. Although these datasets do not directly identify individual passengers, trip times and pickup locations may still raise privacy concerns. To reduce these risks, we will only use the information needed for our experiments, avoid attempting to identify passengers or drivers, and report our results in aggregate. We will also use simulated taxi starting locations rather than actual driver location histories.

Another concern is that traffic data may not cover all roads or neighborhoods equally. Areas with fewer traffic measurements may have less accurate travel-time estimates, which could affect how quickly taxis reach passengers. We will document missing data, use consistent fallback estimates for roads without traffic information, and, if possible, compare results across different geographic areas to identify potential differences in routing performance.

We also recognize that the fastest route is not always the safest or most appropriate route. Although our simulation will follow available road directions and restrictions, reducing pickup waiting time alone does not guarantee passenger or pedestrian safety. Traffic-aware routing could also redirect vehicles through residential streets and potentially increase congestion in those areas. We will discuss these limitations and avoid treating simulation results as proof that an autonomous taxi would be safe or equally effective in real-world conditions.

# Team Expectations

Communication: We will use Discord as our main way to communicate, share updates, and ask questions. Everyone should respond within a reasonable amount of time and let the group know if they need help or are unavailable.

Work Distribution: We will divide tasks fairly based on each person's strengths and interests. Everyone is expected to contribute to the coding, research, or writing and understand the overall project, even if we are responsible for different sections.

Deadlines and Meetings: We will set internal deadlines before the official due dates so we have enough time to review and make changes. We will schedule meetings when needed to discuss our progress, solve problems, and plan our next steps.

Collaboration: We will use our shared GitHub repository to organize code and track changes. Everyone should keep their work updated and communicate before making major changes that could affect other parts of the project.

Decision-Making: We will discuss important decisions as a group and consider everyone's opinions. If we disagree, we will compare the options based on feasibility, project requirements, and available time before making a final decision together.

Handling Difficulties: If someone runs into technical problems or cannot finish their assigned work on time, they should let the group know as early as possible. We will help each other troubleshoot and adjust responsibilities when necessary rather than leaving problems until the last minute.

Final Review: Before submitting the proposal or final report, all three members will review the work to make sure the sections are consistent, the results are accurate, and the project meets the professor's requirements.


**Team Responsibilities:** For the proposal, Charis is responsible for the Abstract, Problem Statement, and Team Expectations; Krasimira is responsible for the Background, Data, and Project Timeline; and Ran is responsible for the Proposed Solution, Evaluation Metrics, and Ethics & Privacy. All three members will review the final proposal together and coordinate the implementation, testing, and final report.

# Project Timeline Proposal

| Meeting Date | Meeting Time | Completed Before Meeting | Discuss At Meeting |
| --- | --- | --- | --- |
| 11/3 | 1 PM | Discuss Project Proposal | Determine best form of communication; Discuss and decide on final project topic; discuss hypothesis; begin background research |
| 11/4 | 6 PM | Final checkin for Project Proposal | Project Proposal due at 11:59 PM<br>Begin building solution |
| 11/11 | 1 PM | Check in for solution building<br>Discuss evaluation metrics | Wrap up solution building<br>Begin diving and writing results report |
| 11/18 | 1 PM | Check in for results report | Begin dividing and writing discussion report |
| 11/25 | 1 PM | Check in for discussion report<br>Revisit project proposal | Revisit project proposal for final project sections<br>Begin wrapping up final project |
| 11/30 | 1 PM | Semi-Final Check In | Discuss/edit full project |
| 12/2 | 1 PM | Final Check in for Final Project | Turn in Final Project before 11:59 PM |

# Footnotes

1. Tsin Yen, K. (2024). Uber and the ride-sharing industry: Overview.: Economics: Research starters: Ebscohost. EBSCO. https://www.ebsco.com/research-starters/economics/uber-and-ride-sharing-industry-overview/

2. Hill, S. (2018, March 27). Ridesharing versus public transit. The American Prospect. https://prospect.org/2018/03/27/ridesharing-versus-public-transit/   

3. Haferkamp, J., Ulmer, M. W., & Ehmke, J. F. (2024). Heatmap-based decision support for repositioning in ride-sharing systems. Transportation Science (INFORMS), 58(1), 110–130. https://doi.org/10.1287/trsc.2023.1202

4. Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A formal basis for the heuristic determination of minimum cost paths. *IEEE Transactions on Systems Science and Cybernetics, 4*(2), 100–107. https://doi.org/10.1109/TSSC.1968.300136